# Does the single-neuron collapse ceiling transfer to a probe direction?

*Leverage Is Not Reach* (arXiv 2606.19831) predicts output collapse at a dose
`t* = m*B`, where `B = ||r_L||/||v||` is the dose at which the write's magnitude
reaches the residual's, and `m* = 1.46` for Llama-3.1-8B (band 1.37–1.65 across
Qwen and Llama, 0.31 for near-rank-one Gemma). The paper's stated scope is
**single-neuron, K=1**, writing along an FFN down-projection column. It names
extension to residual-stream steering vectors as unvalidated future work.

This runs that extension.

**Why the doses are comparable.** Normalize every direction to unit length and the
budget-normalized dose is just the write-to-residual ratio:

`h ← h + t·||r_L||·u` with `||u|| = 1`  ⟹  `||Δh|| / ||r_L|| = t`

So `t` means the same thing for an FFN column and for a difference-of-means probe
direction, and collapse is predicted at `t ≈ 1.46` either way.

**The FFN control is load-bearing.** If a normalized down-projection column does
*not* collapse inside 1.37–1.65 here, this harness has not reproduced their
protocol, the run returns `PROTOCOL_NOT_REPRODUCED`, and no comparison across
intervention classes is licensed.

**No scope task, no rollout, no behavioural outcome.** §14 showed Arm G's dependent
variable is catalogue-position determined and §16 showed its dose ladder never
exceeded a third of the single-layer budget. Neither touches a measurement of when
the model breaks — collapse is a property of the intervention and the model.

For reference, Arm G's own doses converted into these units: removal k=3.0 at 32
layers = **0.166**, addition c=8σ at one layer = **0.490**. Predicted collapse for
the additive arm is **c ≈ 24σ**.

In [ ]:
print("Protocol: ARM_G_CEILING_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-ceiling-launch"
LAUNCH_FILES = (
    "arm_g_ceiling.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")

# The probe direction is read from the seed-112 re-extraction artifact at a
# path relative to the working directory, so it has to land under /content.
ARTIFACT_DIR = "results/arm_g_reextract_seed112_v1"
os.makedirs(f"/content/{ARTIFACT_DIR}", exist_ok=True)
artifact = "arm_g_reextract_result.json.gz.b64"
shutil.copy2(f"{LAUNCH_DIR}/{artifact}", f"/content/{ARTIFACT_DIR}/{artifact}")
assert os.path.exists(f"/content/{ARTIFACT_DIR}/{artifact}")
print("Arm G ceiling launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-ceiling-v1"
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests, no GPU and no network:
#  - a unit direction makes the budget dose equal the write-to-residual ratio
#  - the write hook applies at every position, scalar and per-row
#  - half-max recovers a known cliff, and returns None on a flat curve
#  - all four verdict branches, including the aborting control branch
#  - the section 16 conversion constants, so the framing cannot drift
import os, subprocess, sys
os.chdir("/content")
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_ceiling.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Dose ladder in budget units, forward-only, on generic prompts.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result = json.load(open(f"{WORK_DIR}/arm_g_ceiling_result.json"))
for layer, entry in result["by_layer"].items():
    print(f"=== layer {layer}   ||r_L|| = {entry['residual_norm']:.3f} ===")
    print("  decision:", entry["decision"])
    for reason in entry["decision_reasons"]:
        print("   -", reason)
    print(f"  {'direction':16s} {'collapse dose':>14}  {'vs m*=1.46':>11}")
    for name, dose in entry["collapse_dose"].items():
        if dose is None:
            print(f"  {name:16s} {'none in range':>14}  {'--':>11}")
        else:
            print(f"  {name:16s} {dose:>14.3f}  {dose/1.46:>10.2f}x")
    print()

In [ ]:
# The curves themselves. If the cliff is real it should be visible as a knee,
# and the FFN column and the probe direction should knee in the same place --
# or not, which is the interesting outcome.
DOSES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0]
for layer, entry in result["by_layer"].items():
    print(f"=== layer {layer}: mean next-token entropy by dose ===")
    names = list(entry["collapse_dose"])
    print(f"  {'dose':>5} " + " ".join(f"{n[:13]:>14}" for n in names))
    for dose in DOSES:
        row = " ".join(
            f"{entry['curves'][f'{n}@{dose}']['mean_entropy']:>14.4f}" for n in names
        )
        print(f"  {dose:>5.2f} {row}")
    print(f"  baseline entropy = {entry['curves']['baseline']['mean_entropy']:.4f}")
    print()

In [ ]:
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_ceiling_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_ceiling_result.json.gz.b64"
summary = {
    "config": result["config"],
    "by_layer": {
        layer: {k: v for k, v in entry.items() if k != "curves"}
        for layer, entry in result["by_layer"].items()
    },
}
with open(summary_path, "w") as h:
    json.dump({**summary, "full_result_artifact": "arm_g_ceiling_result.json.gz.b64"},
              h, indent=1)
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(json.dumps(result).encode("utf-8"))))
print("summary:", summary_path)
print("archive:", archive_path)